In [1]:
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt

In [2]:
# TESTING_FOLDER = Path.home() / Path("Desktop/testing")
TESTING_FOLDER = Path("testing/")
METADATA_PATH = TESTING_FOLDER / "metadata.xlsx"

In [3]:
df = pd.read_excel(METADATA_PATH)
id_map = {row.Path: i for i, row in enumerate(df.itertuples(index=False))}
df["id"] = df["Path"].map(id_map)
df

,Path,Description,id
0,1.jpg,"orange cat covers its face with a paw, underne...",0
1,2.jpg,"dog lies in its bed, that is placed on the carpet",1
2,3.jpg,"a tree-shaped-like anthenna, text on the image...",2
3,4.jpg,"a cat with a terrified expression on its face,...",3
4,5.jpg,a drawing of a dog with the bubble over its he...,4
5,6.jpg,"a comic, cat says meow to the woman, woman say...",5
6,7.jpg,image of a man with a mask and a shirt that sa...,6
7,8.jpg,tweet of graph of an lg washing machine intern...,7
8,9.jpg,"""are you winning son?"" meme",8
9,10.jpg,post about computer safety from a trans person,9


In [4]:
TOP_K = 10

In [5]:
def reciprocal_rank(results, correct_id):
    try:
        rank = results.index(correct_id) + 1
        return 1 / rank
    except ValueError:
        return 0.0
    
def hit_at_k(results, correct_id, k):
    return int(correct_id in results[:k])

In [6]:
from cli.platform_file_revealer import PlatformFileRevealer
from cli.table_maker import TableMaker
from infrastructure.services.multimedia_type_detector import MultimediaTypeFinder
from infrastructure.bge_small_encoder import BGEEncoder
from infrastructure.qdrant.qdrant_vdb import QdrantVectorDatabase
from infrastructure.qdrant.qdrant_output_processor import QdrantOutputProcessor
from infrastructure.ml.device import get_device_config
from cli.cli_decoder import CLIDecoder
from settings import *

from dotenv import load_dotenv
import os


load_dotenv("config.env")
hf_token = os.getenv("HF_TOKEN")
config = get_device_config()

encoder = BGEEncoder(hf_token, config)
db = QdrantVectorDatabase()
mtf = MultimediaTypeFinder()

output_processor = QdrantOutputProcessor()
table_maker = TableMaker()
revealer = PlatformFileRevealer()

/Users/williamleonheart/Documents/multimedia-by-prompt-search/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1368.41it/s]


In [ ]:
import pandas as pd

eval_rows = []

for row in df.itertuples(index=False):
    target_id = row.id
    query_text = row.Description

    embedding = encoder.encode(query_text)
    results = db.search(embedding, TOP_K)

    rows = output_processor.create_dict_of_rows(results)

    retrieved_ids = []
    for r in rows:
        path = r.get("path")
        file_id = id_map.get(Path(path).name) if path else None
        retrieved_ids.append(file_id)

    rr = reciprocal_rank(retrieved_ids, target_id)

    eval_rows.append({
        "query": query_text,
        "target_id": target_id,
        "rr": rr,
        "hit@1": int(target_id in retrieved_ids[:1]),
        "hit@2": int(target_id in retrieved_ids[:2]),
        "hit@3": int(target_id in retrieved_ids[:3]),
        "hit@4": int(target_id in retrieved_ids[:4]),
    })

eval_df = pd.DataFrame(eval_rows)

In [9]:
db.client.close()

In [13]:
eval_df

,query,target_id,rr,hit@1,hit@2,hit@3,hit@4
0,"orange cat covers its face with a paw, underne...",0,0.500000,0,1,1,1
1,"dog lies in its bed, that is placed on the carpet",1,1.000000,1,1,1,1
2,"a tree-shaped-like anthenna, text on the image...",2,1.000000,1,1,1,1
3,"a cat with a terrified expression on its face,...",3,1.000000,1,1,1,1
4,a drawing of a dog with the bubble over its he...,4,0.500000,0,1,1,1
5,"a comic, cat says meow to the woman, woman say...",5,1.000000,1,1,1,1
6,image of a man with a mask and a shirt that sa...,6,0.333333,0,0,1,1
7,tweet of graph of an lg washing machine intern...,7,1.000000,1,1,1,1
8,"""are you winning son?"" meme",8,0.000000,0,0,0,0
9,post about computer safety from a trans person,9,0.125000,0,0,0,0
